### CNN trained on data encoded with scaling factor 0.5

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score 
import json

Load the labels

In [2]:
def load_labels(file_path):
    with open(file_path, 'r') as file:
        labels = file.readlines()
    labels = np.array([int(label.strip()) for label in labels])
    return labels

Load the images and labels into arrays

In [ ]:
def load_data(image_folder, labels_file):
    labels = load_labels(labels_file)
    image_size = (124, 124)  # Resize to a smaller image size
    images = []
    
    for img_name in sorted(os.listdir(image_folder)):
        img_path = os.path.join(image_folder, img_name)
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=image_size, color_mode='grayscale')
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        images.append(img_array)
    
    images = np.array(images)
    return images, labels


CNN model

In [ ]:
def build_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),

        Conv2D(32, kernel_size=(3, 3), kernel_regularizer=l2(0.001), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(64, kernel_size=(3, 3), kernel_regularizer=l2(0.001), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),

        Flatten(),

        Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ]) 

    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

    return model

Cross-validation

In [ ]:
def cross_validate_model(image_folder, labels_file, num_folds=10):
    # Load the dataset
    images, labels = load_data(image_folder, labels_file)
    
    # Normalize images
    images = images.astype('float32') / 255.0

    X_trainval, X_test, y_trainval, y_test = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)

    # Define KFold cross-validation on the train set
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

    fold_no = 1
    accuracy_per_fold = []
    auc_per_fold = []
    all_classification_reports = []
    all_auc_scores = []

    for train_index, val_index in kf.split(X_trainval):  # KFold only on train set
        X_train_fold, X_val_fold = X_trainval[train_index], X_trainval[val_index]
        y_train_fold, y_val_fold = y_trainval[train_index], y_trainval[val_index]

        # Build and compile the model
        input_shape = X_trainval.shape[1:]
        model = build_cnn_model(input_shape)

        print(f'\nTraining fold {fold_no}...')

        # Train the model
        model.fit(X_train_fold, y_train_fold , validation_data=(X_val_fold, y_val_fold), epochs=15, batch_size=8, verbose=1)

        # Evaluate on the test set
        y_pred_prob = model.predict(X_test)

        y_pred = (y_pred_prob > 0.5).astype(int)

        # Calculate accuracy
        accuracy = accuracy_score(y_test, y_pred) 
        accuracy_per_fold.append(accuracy)

        # Calculate AUC
        auc = roc_auc_score(y_test, y_pred_prob)
        auc_per_fold.append(auc)
        all_auc_scores.append(auc)

        # Classification Report
        class_report = classification_report(y_test, y_pred, target_names=['Negative', 'Positive'], output_dict=True, zero_division=0)
        all_classification_reports.append(class_report)

        print(f'Fold {fold_no} accuracy: {accuracy:.4f}')
        print(f'Fold {fold_no} AUC: {auc:.4f}')

        fold_no += 1

    # Average results
    avg_accuracy = np.mean(accuracy_per_fold)
    std_accuracy = np.std(accuracy_per_fold)
    avg_auc = np.mean(auc_per_fold)
    std_auc = np.std(auc_per_fold)

    # Average classification metrics
    avg_classification_report = {
        'Positive': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Positive']['precision'] for r in all_classification_reports])),
            'recall': float(np.mean([r['Positive']['recall'] for r in all_classification_reports])),
            'f1-score': float(np.mean([r['Positive']['f1-score'] for r in all_classification_reports]))
        },
        'Negative': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Negative']['precision'] for r in all_classification_reports])),
            'recall': float(np.mean([r['Negative']['recall'] for r in all_classification_reports])),
            'f1-score': float(np.mean([r['Negative']['f1-score'] for r in all_classification_reports]))
        }
    }

    print("\nAverage Classification Report (across all folds):")
    print(avg_classification_report)

    return accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report


Saving the results into a JSON

In [7]:
def save_classification_report(dataset_name, avg_classification_report):
    results_file = "reports/classification_reports3_cnn.json"

    # Add dataset name to the report
    report_to_save = {
        "dataset": dataset_name,
        "report": avg_classification_report
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(report_to_save)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Classification report saved to {results_file}")

In [8]:
def save_results(dataset_name, accuracy_per_fold, auc_per_fold, all_classification_reports):
    results_file = "reports/classification_results3_CNN.json"

    # Convert results to a dictionary
    results_dict = {
        "dataset": dataset_name,
        "accuracies": accuracy_per_fold,
        "roc_aucs": auc_per_fold,
        "classification_report": all_classification_reports
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(results_dict)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Results saved to {results_file}")



File paths RES 25

In [24]:
image_folder_antiinflam = 'data/images/img_res25/aip_antiinflam' 
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [25]:
image_folder_antipb = 'data/images/img_res25/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [26]:
image_folder_antipb2 = 'data/images/img_res25/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [27]:
image_folder_csamp = 'data/images/img_res25/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [28]:
image_folder_hivddi = 'data/images/img_res25/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [29]:
image_folder_hivrtv = 'data/images/img_res25/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 25

In [30]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10 )


Training fold 1...
Epoch 1/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 49s 237ms/step - accuracy: 0.5722 - loss: 0.8594 - val_accuracy: 0.6294 - val_loss: 0.7004
Epoch 2/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 45s 232ms/step - accuracy: 0.6587 - loss: 0.6789 - val_accuracy: 0.6529 - val_loss: 0.6751
Epoch 3/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 82s 231ms/step - accuracy: 0.7184 - loss: 0.6299 - val_accuracy: 0.6824 - val_loss: 0.6565
Epoch 4/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 47s 246ms/step - accuracy: 0.7650 - loss: 0.5679 - val_accuracy: 0.7118 - val_loss: 0.6465
Epoch 5/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 80s 234ms/step - accuracy: 0.7624 - loss: 0.5569 - val_accuracy: 0.7176 - val_loss: 0.6455
Epoch 6/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 46s 241ms/step - accuracy: 0.7888 - loss: 0.5212 - val_accuracy: 0.7118 - val_loss: 0.7020
Epoch 7/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 78s 219ms/step - accuracy: 0.7936 - loss: 0.5252 - val_accuracy: 0.7294 - val_loss: 0.6762
Epoch 8/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 86s 239ms/step - accura

In [33]:
save_results("aip_antiinflam_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_25", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [34]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10 )


Training fold 1...
Epoch 1/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 75s 907ms/step - accuracy: 0.5394 - loss: 0.9159 - val_accuracy: 0.5652 - val_loss: 0.7481
Epoch 2/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 29s 224ms/step - accuracy: 0.6100 - loss: 0.7306 - val_accuracy: 0.8696 - val_loss: 0.5948
Epoch 3/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 18s 228ms/step - accuracy: 0.7359 - loss: 0.6307 - val_accuracy: 0.9130 - val_loss: 0.5086
Epoch 4/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 19s 248ms/step - accuracy: 0.8170 - loss: 0.5045 - val_accuracy: 0.8116 - val_loss: 0.5318
Epoch 5/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 19s 249ms/step - accuracy: 0.8402 - loss: 0.4847 - val_accuracy: 0.8261 - val_loss: 0.4899
Epoch 6/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 21s 250ms/step - accuracy: 0.8605 - loss: 0.4244 - val_accuracy: 0.8841 - val_loss: 0.4586
Epoch 7/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 21s 257ms/step - accuracy: 0.8806 - loss: 0.4258 - val_accuracy: 0.8551 - val_loss: 0.4497
Epoch 8/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 20s 252ms/step - accuracy: 0.8759 - los

In [35]:
save_results("amp_antibp_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_25", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [36]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10 )


Training fold 1...
Epoch 1/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 52s 257ms/step - accuracy: 0.5255 - loss: 0.8940 - val_accuracy: 0.8000 - val_loss: 0.6309
Epoch 2/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 49s 275ms/step - accuracy: 0.7250 - loss: 0.6164 - val_accuracy: 0.8062 - val_loss: 0.5366
Epoch 3/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 48s 266ms/step - accuracy: 0.7982 - loss: 0.5385 - val_accuracy: 0.7750 - val_loss: 0.5214
Epoch 4/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 44s 245ms/step - accuracy: 0.7974 - loss: 0.5088 - val_accuracy: 0.7688 - val_loss: 0.5564
Epoch 5/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 46s 258ms/step - accuracy: 0.8214 - loss: 0.4975 - val_accuracy: 0.7937 - val_loss: 0.5155
Epoch 6/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 82s 257ms/step - accuracy: 0.8328 - loss: 0.4721 - val_accuracy: 0.8125 - val_loss: 0.4949
Epoch 7/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 82s 253ms/step - accuracy: 0.8218 - loss: 0.4456 - val_accuracy: 0.7937 - val_loss: 0.4997
Epoch 8/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 80s 243ms/step - accura

In [37]:
save_results("amp_antibp2_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_25", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [38]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 226ms/step - accuracy: 0.4238 - loss: 1.2534 - val_accuracy: 0.5238 - val_loss: 0.8152
Epoch 2/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 202ms/step - accuracy: 0.4685 - loss: 0.8238 - val_accuracy: 0.7143 - val_loss: 0.7937
Epoch 3/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 207ms/step - accuracy: 0.5403 - loss: 0.7889 - val_accuracy: 0.5238 - val_loss: 0.7615
Epoch 4/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 202ms/step - accuracy: 0.6043 - loss: 0.7595 - val_accuracy: 0.7619 - val_loss: 0.7172
Epoch 5/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 220ms/step - accuracy: 0.7178 - loss: 0.7165 - val_accuracy: 0.5714 - val_loss: 0.7043
Epoch 6/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 212ms/step - accuracy: 0.6854 - loss: 0.6685 - val_accuracy: 0.7143 - val_loss: 0.6385
Epoch 7/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 205ms/step - accuracy: 0.6859 - loss: 0.6176 - val_accuracy: 0.7619 - val_loss: 0.6136
Epoch 8/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 213ms/step - accuracy: 0.7668 - loss: 0.582

In [39]:
save_results("amp_csamp_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_25", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [40]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 15s 216ms/step - accuracy: 0.4426 - loss: 1.2200 - val_accuracy: 0.4200 - val_loss: 0.7849
Epoch 2/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 207ms/step - accuracy: 0.5104 - loss: 0.7763 - val_accuracy: 0.4200 - val_loss: 0.7588
Epoch 3/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 209ms/step - accuracy: 0.5581 - loss: 0.7523 - val_accuracy: 0.4800 - val_loss: 0.7441
Epoch 4/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 206ms/step - accuracy: 0.5127 - loss: 0.7427 - val_accuracy: 0.5800 - val_loss: 0.7342
Epoch 5/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 208ms/step - accuracy: 0.5391 - loss: 0.7326 - val_accuracy: 0.6200 - val_loss: 0.7283
Epoch 6/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 211ms/step - accuracy: 0.5525 - loss: 0.7275 - val_accuracy: 0.6000 - val_loss: 0.7242
Epoch 7/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 208ms/step - accuracy: 0.5119 - loss: 0.7247 - val_accuracy: 0.6400 - val_loss: 0.7195
Epoch 8/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 218ms/step - accuracy: 0.5590 - los

In [41]:
save_results("hiv_ddi_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_25", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [42]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10)


Training fold 1...
Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 202ms/step - accuracy: 0.4932 - loss: 0.9656 - val_accuracy: 0.6441 - val_loss: 0.7682
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 207ms/step - accuracy: 0.5817 - loss: 0.7648 - val_accuracy: 0.5593 - val_loss: 0.7414
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 208ms/step - accuracy: 0.6356 - loss: 0.7283 - val_accuracy: 0.4407 - val_loss: 0.7800
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 212ms/step - accuracy: 0.5846 - loss: 0.7317 - val_accuracy: 0.6441 - val_loss: 0.6946
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 208ms/step - accuracy: 0.6610 - loss: 0.6554 - val_accuracy: 0.7458 - val_loss: 0.6473
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 13s 198ms/step - accuracy: 0.7681 - loss: 0.5915 - val_accuracy: 0.7627 - val_loss: 0.5961
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 13s 197ms/step - accuracy: 0.7999 - loss: 0.5101 - val_accuracy: 0.7458 - val_loss: 0.6092
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 21s 200ms/step - accuracy: 0.7804 - los

In [43]:
save_results("hiv_rtv_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_25", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


Files paths RES 50


In [44]:
image_folder_antiinflam = 'data/images/aip_antiinflam'
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [45]:
image_folder_antipb = 'data/images/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [46]:
image_folder_antipb2 = 'data/images/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [47]:
image_folder_csamp = 'data/images/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [48]:
image_folder_hivddi = 'data/images/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [49]:
image_folder_hivrtv = 'data/images/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 50

In [50]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10)


Training fold 1...
Epoch 1/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 42s 208ms/step - accuracy: 0.5584 - loss: 0.9800 - val_accuracy: 0.6176 - val_loss: 0.7364
Epoch 2/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 39s 203ms/step - accuracy: 0.5672 - loss: 0.7283 - val_accuracy: 0.6176 - val_loss: 0.6985
Epoch 3/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 46s 240ms/step - accuracy: 0.5978 - loss: 0.6986 - val_accuracy: 0.6176 - val_loss: 0.6743
Epoch 4/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 81s 233ms/step - accuracy: 0.6091 - loss: 0.6693 - val_accuracy: 0.6294 - val_loss: 0.6434
Epoch 5/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 45s 234ms/step - accuracy: 0.6743 - loss: 0.6164 - val_accuracy: 0.6706 - val_loss: 0.6288
Epoch 6/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 43s 226ms/step - accuracy: 0.7101 - loss: 0.5943 - val_accuracy: 0.6647 - val_loss: 0.6153
Epoch 7/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 42s 218ms/step - accuracy: 0.7073 - loss: 0.5810 - val_accuracy: 0.6471 - val_loss: 0.6414
Epoch 8/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 45s 234ms/step - accura

In [51]:
save_results("aip_antiinflam_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_50", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [52]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10) 


Training fold 1...
Epoch 1/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 23s 231ms/step - accuracy: 0.4684 - loss: 1.1236 - val_accuracy: 0.4783 - val_loss: 0.7902
Epoch 2/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 17s 220ms/step - accuracy: 0.5200 - loss: 0.7824 - val_accuracy: 0.4783 - val_loss: 0.7633
Epoch 3/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 17s 214ms/step - accuracy: 0.5167 - loss: 0.7599 - val_accuracy: 0.4783 - val_loss: 0.7514
Epoch 4/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 17s 215ms/step - accuracy: 0.5542 - loss: 0.7455 - val_accuracy: 0.6522 - val_loss: 0.7372
Epoch 5/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 16s 202ms/step - accuracy: 0.5675 - loss: 0.7355 - val_accuracy: 0.7826 - val_loss: 0.7281
Epoch 6/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 16s 204ms/step - accuracy: 0.6004 - loss: 0.7250 - val_accuracy: 0.7246 - val_loss: 0.7208
Epoch 7/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 16s 201ms/step - accuracy: 0.6177 - loss: 0.7130 - val_accuracy: 0.8406 - val_loss: 0.6910
Epoch 8/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 17s 212ms/step - accuracy: 0.6615 - los

In [53]:
save_results("amp_antibp_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_50", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [54]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10)


Training fold 1...
Epoch 1/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 40s 202ms/step - accuracy: 0.5067 - loss: 0.8801 - val_accuracy: 0.4625 - val_loss: 0.7398
Epoch 2/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 37s 204ms/step - accuracy: 0.5322 - loss: 0.7255 - val_accuracy: 0.6313 - val_loss: 0.7153
Epoch 3/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 35s 196ms/step - accuracy: 0.5971 - loss: 0.7162 - val_accuracy: 0.5688 - val_loss: 0.6760
Epoch 4/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 35s 197ms/step - accuracy: 0.6714 - loss: 0.6592 - val_accuracy: 0.7625 - val_loss: 0.5387
Epoch 5/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 37s 205ms/step - accuracy: 0.7592 - loss: 0.5517 - val_accuracy: 0.7937 - val_loss: 0.5085
Epoch 6/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 38s 211ms/step - accuracy: 0.7841 - loss: 0.5137 - val_accuracy: 0.8000 - val_loss: 0.4923
Epoch 7/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 37s 206ms/step - accuracy: 0.8294 - loss: 0.4805 - val_accuracy: 0.7812 - val_loss: 0.4922
Epoch 8/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 35s 194ms/step - accura

In [55]:
save_results("amp_antibp2_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_50", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [56]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 280ms/step - accuracy: 0.4407 - loss: 1.1373 - val_accuracy: 0.4762 - val_loss: 0.7968
Epoch 2/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 213ms/step - accuracy: 0.4833 - loss: 0.8047 - val_accuracy: 0.4762 - val_loss: 0.7784
Epoch 3/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 211ms/step - accuracy: 0.5145 - loss: 0.7742 - val_accuracy: 0.5238 - val_loss: 0.7618
Epoch 4/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 223ms/step - accuracy: 0.5850 - loss: 0.7599 - val_accuracy: 0.5238 - val_loss: 0.7547
Epoch 5/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 225ms/step - accuracy: 0.4594 - loss: 0.7643 - val_accuracy: 0.7143 - val_loss: 0.7442
Epoch 6/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 231ms/step - accuracy: 0.5054 - loss: 0.7472 - val_accuracy: 0.7143 - val_loss: 0.7324
Epoch 7/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 211ms/step - accuracy: 0.7429 - loss: 0.7332 - val_accuracy: 0.7143 - val_loss: 0.7261
Epoch 8/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 216ms/step - accuracy: 0.6339 - loss: 0.72

In [57]:
save_results("amp_csamp_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_50", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [58]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 15s 220ms/step - accuracy: 0.5026 - loss: 0.9442 - val_accuracy: 0.4200 - val_loss: 0.7750
Epoch 2/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 209ms/step - accuracy: 0.5171 - loss: 0.7660 - val_accuracy: 0.4200 - val_loss: 0.7507
Epoch 3/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 206ms/step - accuracy: 0.4697 - loss: 0.7476 - val_accuracy: 0.5800 - val_loss: 0.7366
Epoch 4/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 211ms/step - accuracy: 0.4786 - loss: 0.7358 - val_accuracy: 0.4200 - val_loss: 0.7291
Epoch 5/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 215ms/step - accuracy: 0.5238 - loss: 0.7270 - val_accuracy: 0.4800 - val_loss: 0.7224
Epoch 6/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 209ms/step - accuracy: 0.4703 - loss: 0.7214 - val_accuracy: 0.4200 - val_loss: 0.7416
Epoch 7/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 211ms/step - accuracy: 0.4963 - loss: 0.7225 - val_accuracy: 0.5800 - val_loss: 0.7143
Epoch 8/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 209ms/step - accuracy: 0.5183 - los

In [59]:
save_results("hiv_ddi_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_50", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [60]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10 )


Training fold 1...
Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 177ms/step - accuracy: 0.4780 - loss: 1.0680 - val_accuracy: 0.5593 - val_loss: 0.7814
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 172ms/step - accuracy: 0.5089 - loss: 0.7752 - val_accuracy: 0.5593 - val_loss: 0.7558
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 21s 178ms/step - accuracy: 0.5227 - loss: 0.7525 - val_accuracy: 0.5593 - val_loss: 0.7411
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 182ms/step - accuracy: 0.4892 - loss: 0.7416 - val_accuracy: 0.5593 - val_loss: 0.7318
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 183ms/step - accuracy: 0.5337 - loss: 0.7306 - val_accuracy: 0.5593 - val_loss: 0.7241
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 182ms/step - accuracy: 0.4918 - loss: 0.7281 - val_accuracy: 0.5593 - val_loss: 0.7208
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 188ms/step - accuracy: 0.4902 - loss: 0.7217 - val_accuracy: 0.5593 - val_loss: 0.7176
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 186ms/step - accuracy: 0.5369 - los

In [61]:
save_results("hiv_rtv_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_50", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


File paths RES 75

In [62]:
image_folder_antiinflam = 'data/images/img_res75/aip_antiinflam' 
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [63]:
image_folder_antipb = 'data/images/img_res75/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [64]:
image_folder_antipb2 = 'data/images/img_res75/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [65]:
image_folder_csamp = 'data/images/img_res75/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [66]:
image_folder_hivddi = 'data/images/img_res75/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [9]:
image_folder_hivrtv = 'data/images/img_res75/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 75

In [68]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10)


Training fold 1...
Epoch 1/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 38s 185ms/step - accuracy: 0.5897 - loss: 0.8127 - val_accuracy: 0.6176 - val_loss: 0.7172
Epoch 2/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 37s 193ms/step - accuracy: 0.5813 - loss: 0.7143 - val_accuracy: 0.6176 - val_loss: 0.6828
Epoch 3/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 35s 181ms/step - accuracy: 0.5886 - loss: 0.7001 - val_accuracy: 0.6176 - val_loss: 0.6837
Epoch 4/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 34s 176ms/step - accuracy: 0.6042 - loss: 0.6890 - val_accuracy: 0.6176 - val_loss: 0.6734
Epoch 5/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 35s 181ms/step - accuracy: 0.5782 - loss: 0.6820 - val_accuracy: 0.6353 - val_loss: 0.6455
Epoch 6/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 34s 176ms/step - accuracy: 0.6313 - loss: 0.6372 - val_accuracy: 0.6706 - val_loss: 0.6254
Epoch 7/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 34s 176ms/step - accuracy: 0.6627 - loss: 0.5967 - val_accuracy: 0.6529 - val_loss: 0.6256
Epoch 8/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 34s 178ms/step - accura

In [69]:
save_results("aip_antiinflam_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_75", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [70]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report= cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10)


Training fold 1...
Epoch 1/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 18s 199ms/step - accuracy: 0.4623 - loss: 0.9786 - val_accuracy: 0.5217 - val_loss: 0.7629
Epoch 2/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 190ms/step - accuracy: 0.5177 - loss: 0.7565 - val_accuracy: 0.4783 - val_loss: 0.7409
Epoch 3/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 192ms/step - accuracy: 0.5074 - loss: 0.7386 - val_accuracy: 0.4783 - val_loss: 0.7293
Epoch 4/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 190ms/step - accuracy: 0.4886 - loss: 0.7272 - val_accuracy: 0.4783 - val_loss: 0.7220
Epoch 5/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 191ms/step - accuracy: 0.5177 - loss: 0.7213 - val_accuracy: 0.4783 - val_loss: 0.7174
Epoch 6/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.5292 - loss: 0.7161 - val_accuracy: 0.4783 - val_loss: 0.7137
Epoch 7/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.4629 - loss: 0.7132 - val_accuracy: 0.4783 - val_loss: 0.7110
Epoch 8/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.5265 - los

In [71]:
save_results("amp_antibp_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_75", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [72]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10)


Training fold 1...
Epoch 1/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 37s 189ms/step - accuracy: 0.5360 - loss: 0.9595 - val_accuracy: 0.4688 - val_loss: 0.7366
Epoch 2/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 33s 182ms/step - accuracy: 0.5903 - loss: 0.7264 - val_accuracy: 0.7625 - val_loss: 0.6707
Epoch 3/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 33s 183ms/step - accuracy: 0.6665 - loss: 0.6585 - val_accuracy: 0.8062 - val_loss: 0.5459
Epoch 4/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 32s 177ms/step - accuracy: 0.7292 - loss: 0.6017 - val_accuracy: 0.8125 - val_loss: 0.5310
Epoch 5/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 32s 180ms/step - accuracy: 0.7511 - loss: 0.5752 - val_accuracy: 0.7812 - val_loss: 0.5061
Epoch 6/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 32s 176ms/step - accuracy: 0.7982 - loss: 0.5141 - val_accuracy: 0.7688 - val_loss: 0.5043
Epoch 7/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 33s 181ms/step - accuracy: 0.8129 - loss: 0.4898 - val_accuracy: 0.7688 - val_loss: 0.5045
Epoch 8/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 32s 176ms/step - accura

In [73]:
save_results("amp_antibp2_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_75", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [74]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 7s 200ms/step - accuracy: 0.5286 - loss: 1.2205 - val_accuracy: 0.4762 - val_loss: 0.8362
Epoch 2/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 186ms/step - accuracy: 0.5212 - loss: 0.8181 - val_accuracy: 0.5238 - val_loss: 0.8046
Epoch 3/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 189ms/step - accuracy: 0.5754 - loss: 0.7987 - val_accuracy: 0.4762 - val_loss: 0.7875
Epoch 4/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 184ms/step - accuracy: 0.5131 - loss: 0.7834 - val_accuracy: 0.4762 - val_loss: 0.7738
Epoch 5/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 188ms/step - accuracy: 0.5787 - loss: 0.7680 - val_accuracy: 0.5238 - val_loss: 0.7615
Epoch 6/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 184ms/step - accuracy: 0.4638 - loss: 0.7659 - val_accuracy: 0.6190 - val_loss: 0.7508
Epoch 7/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 183ms/step - accuracy: 0.6155 - loss: 0.7474 - val_accuracy: 0.5238 - val_loss: 0.7444
Epoch 8/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 189ms/step - accuracy: 0.4729 - loss: 0.766

In [75]:
save_results("amp_csamp_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_75", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [76]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 12s 186ms/step - accuracy: 0.4811 - loss: 1.0294 - val_accuracy: 0.4200 - val_loss: 0.7733
Epoch 2/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 183ms/step - accuracy: 0.5539 - loss: 0.7633 - val_accuracy: 0.4200 - val_loss: 0.7533
Epoch 3/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 175ms/step - accuracy: 0.5071 - loss: 0.7449 - val_accuracy: 0.4200 - val_loss: 0.7365
Epoch 4/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 177ms/step - accuracy: 0.4442 - loss: 0.7349 - val_accuracy: 0.5800 - val_loss: 0.7276
Epoch 5/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 184ms/step - accuracy: 0.5151 - loss: 0.7269 - val_accuracy: 0.5800 - val_loss: 0.7221
Epoch 6/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 174ms/step - accuracy: 0.4820 - loss: 0.7218 - val_accuracy: 0.5800 - val_loss: 0.7185
Epoch 7/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 174ms/step - accuracy: 0.4585 - loss: 0.7181 - val_accuracy: 0.5800 - val_loss: 0.7153
Epoch 8/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 174ms/step - accuracy: 0.5321 - los

In [ ]:
save_results("hiv_ddi_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_75", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


: 

In [10]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report= cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10)


Training fold 1...
Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 16s 195ms/step - accuracy: 0.5124 - loss: 0.9915 - val_accuracy: 0.5593 - val_loss: 0.7571
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 186ms/step - accuracy: 0.5325 - loss: 0.7516 - val_accuracy: 0.5424 - val_loss: 0.7372
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 184ms/step - accuracy: 0.5151 - loss: 0.7344 - val_accuracy: 0.5593 - val_loss: 0.7254
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 21s 194ms/step - accuracy: 0.4622 - loss: 0.7246 - val_accuracy: 0.5593 - val_loss: 0.7150
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 13s 198ms/step - accuracy: 0.4986 - loss: 0.7213 - val_accuracy: 0.5593 - val_loss: 0.7128
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 20s 195ms/step - accuracy: 0.5374 - loss: 0.7090 - val_accuracy: 0.5593 - val_loss: 0.7125
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 13s 196ms/step - accuracy: 0.4860 - loss: 0.7120 - val_accuracy: 0.5593 - val_loss: 0.7098
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 14s 213ms/step - accuracy: 0.4685 - los

In [11]:
save_results("hiv_rtv_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_75", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


File paths RES 100

In [12]:
image_folder_antiinflam = 'data/images/img_res100/aip_antiinflam'
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [13]:
image_folder_antipb = 'data/images/img_res100/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [14]:
image_folder_antipb2 = 'data/images/img_res100/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [15]:
image_folder_csamp = 'data/images/img_res100/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [16]:
image_folder_hivddi = 'data/images/img_res100/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [17]:
image_folder_hivrtv = 'data/images/img_res100/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 100

In [18]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10)


Training fold 1...
Epoch 1/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 39s 191ms/step - accuracy: 0.5442 - loss: 0.9359 - val_accuracy: 0.6176 - val_loss: 0.7179
Epoch 2/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 36s 187ms/step - accuracy: 0.5833 - loss: 0.7284 - val_accuracy: 0.6176 - val_loss: 0.6999
Epoch 3/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 37s 192ms/step - accuracy: 0.6169 - loss: 0.6983 - val_accuracy: 0.6176 - val_loss: 0.6965
Epoch 4/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 35s 181ms/step - accuracy: 0.5900 - loss: 0.7000 - val_accuracy: 0.6176 - val_loss: 0.6860
Epoch 5/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 35s 180ms/step - accuracy: 0.6038 - loss: 0.6951 - val_accuracy: 0.6176 - val_loss: 0.6817
Epoch 6/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 34s 177ms/step - accuracy: 0.5901 - loss: 0.6877 - val_accuracy: 0.6176 - val_loss: 0.6775
Epoch 7/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 36s 187ms/step - accuracy: 0.5995 - loss: 0.6912 - val_accuracy: 0.6176 - val_loss: 0.6774
Epoch 8/15
192/192 ━━━━━━━━━━━━━━━━━━━━ 36s 190ms/step - accura

In [19]:
save_results("aip_antiinflam_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_100", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [20]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10)


Training fold 1...
Epoch 1/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 16s 177ms/step - accuracy: 0.5220 - loss: 0.9947 - val_accuracy: 0.4783 - val_loss: 0.7750
Epoch 2/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 14s 179ms/step - accuracy: 0.5160 - loss: 0.7677 - val_accuracy: 0.4783 - val_loss: 0.7518
Epoch 3/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 14s 183ms/step - accuracy: 0.5426 - loss: 0.7471 - val_accuracy: 0.4783 - val_loss: 0.7385
Epoch 4/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 21s 183ms/step - accuracy: 0.4801 - loss: 0.7389 - val_accuracy: 0.4783 - val_loss: 0.7284
Epoch 5/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 14s 183ms/step - accuracy: 0.4752 - loss: 0.7266 - val_accuracy: 0.5217 - val_loss: 0.7212
Epoch 6/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 15s 187ms/step - accuracy: 0.4543 - loss: 0.7202 - val_accuracy: 0.4783 - val_loss: 0.7168
Epoch 7/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 14s 185ms/step - accuracy: 0.4794 - loss: 0.7159 - val_accuracy: 0.4783 - val_loss: 0.7134
Epoch 8/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 14s 185ms/step - accuracy: 0.5212 - los

In [21]:
save_results("amp_antibp_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_100", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [22]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10)


Training fold 1...
Epoch 1/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 36s 182ms/step - accuracy: 0.4926 - loss: 0.9375 - val_accuracy: 0.4625 - val_loss: 0.7488
Epoch 2/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 40s 174ms/step - accuracy: 0.5121 - loss: 0.7369 - val_accuracy: 0.4625 - val_loss: 0.7240
Epoch 3/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 32s 176ms/step - accuracy: 0.4991 - loss: 0.7204 - val_accuracy: 0.4625 - val_loss: 0.7147
Epoch 4/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 31s 172ms/step - accuracy: 0.5266 - loss: 0.7122 - val_accuracy: 0.4625 - val_loss: 0.7094
Epoch 5/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 32s 178ms/step - accuracy: 0.4985 - loss: 0.7082 - val_accuracy: 0.4625 - val_loss: 0.7061
Epoch 6/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 31s 174ms/step - accuracy: 0.5066 - loss: 0.7050 - val_accuracy: 0.4625 - val_loss: 0.7041
Epoch 7/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 32s 175ms/step - accuracy: 0.5122 - loss: 0.7031 - val_accuracy: 0.4625 - val_loss: 0.7026
Epoch 8/15
180/180 ━━━━━━━━━━━━━━━━━━━━ 31s 170ms/step - accura

In [23]:
save_results("amp_antibp2_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_100", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [24]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 194ms/step - accuracy: 0.5242 - loss: 1.1047 - val_accuracy: 0.5238 - val_loss: 0.8001
Epoch 2/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 175ms/step - accuracy: 0.4398 - loss: 0.7986 - val_accuracy: 0.5238 - val_loss: 0.7817
Epoch 3/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 181ms/step - accuracy: 0.5793 - loss: 0.7775 - val_accuracy: 0.5238 - val_loss: 0.7681
Epoch 4/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 177ms/step - accuracy: 0.5154 - loss: 0.7661 - val_accuracy: 0.5238 - val_loss: 0.7584
Epoch 5/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 197ms/step - accuracy: 0.5366 - loss: 0.7565 - val_accuracy: 0.5238 - val_loss: 0.7515
Epoch 6/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 189ms/step - accuracy: 0.4246 - loss: 0.7508 - val_accuracy: 0.5238 - val_loss: 0.7545
Epoch 7/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 174ms/step - accuracy: 0.4458 - loss: 0.7675 - val_accuracy: 0.4762 - val_loss: 0.7413
Epoch 8/15
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 180ms/step - accuracy: 0.5195 - loss: 0.739

In [25]:
save_results("amp_csamp_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_100", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [26]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 13s 187ms/step - accuracy: 0.5356 - loss: 1.0091 - val_accuracy: 0.4200 - val_loss: 0.7810
Epoch 2/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 183ms/step - accuracy: 0.4791 - loss: 0.7726 - val_accuracy: 0.5800 - val_loss: 0.7509
Epoch 3/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 11s 186ms/step - accuracy: 0.5086 - loss: 0.7472 - val_accuracy: 0.5800 - val_loss: 0.7366
Epoch 4/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 184ms/step - accuracy: 0.5088 - loss: 0.7352 - val_accuracy: 0.5800 - val_loss: 0.7290
Epoch 5/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 183ms/step - accuracy: 0.5473 - loss: 0.7272 - val_accuracy: 0.4200 - val_loss: 0.7230
Epoch 6/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 180ms/step - accuracy: 0.4779 - loss: 0.7215 - val_accuracy: 0.4200 - val_loss: 0.7190
Epoch 7/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 186ms/step - accuracy: 0.5287 - loss: 0.7168 - val_accuracy: 0.5800 - val_loss: 0.7143
Epoch 8/15
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 181ms/step - accuracy: 0.5140 - los

In [27]:
save_results("hiv_ddi_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_100", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json


In [28]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10)


Training fold 1...
Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.5306 - loss: 1.0833 - val_accuracy: 0.5593 - val_loss: 0.7881
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 177ms/step - accuracy: 0.4891 - loss: 0.7820 - val_accuracy: 0.5593 - val_loss: 0.7597
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 172ms/step - accuracy: 0.5415 - loss: 0.7548 - val_accuracy: 0.5593 - val_loss: 0.7415
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 174ms/step - accuracy: 0.5102 - loss: 0.7415 - val_accuracy: 0.5593 - val_loss: 0.7321
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 175ms/step - accuracy: 0.5117 - loss: 0.7326 - val_accuracy: 0.5593 - val_loss: 0.7248
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 178ms/step - accuracy: 0.5111 - loss: 0.7255 - val_accuracy: 0.5593 - val_loss: 0.7198
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 179ms/step - accuracy: 0.5073 - loss: 0.7204 - val_accuracy: 0.5593 - val_loss: 0.7155
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 12s 176ms/step - accuracy: 0.5053 - los

In [29]:
save_results("hiv_rtv_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_100", avg_classification_report)

Results saved to reports/classification_results3_CNN.json
Classification report saved to reports/classification_reports3_cnn.json
